# 02 - Certificate collection (long-running, resumable)

**Start this first and let it run while everything else is developed.** It is the only step whose duration is not under our control, and every downstream decision depends on the coverage actually achieved.

Safe to interrupt: the ledger is the state. Re-running resumes where it stopped.

The ledger lives on `/content`, not Drive - Drive's FUSE layer does not implement POSIX locking correctly and SQLite depends on it.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive; drive.mount('/content/drive')

REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}

import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd
from src.collect.ledger import Ledger
from src.collect import tls_prober
from src.utils.logging_setup import get_logger

log = get_logger('collect_tls', P['artifacts']['logs'])
ledger = Ledger(P['local']['ledger'],
                drive_backup=f"{P['artifacts']['logs']}/certificate_ledger_backup.db")
ledger.restore_from_backup()   # rebuild after a fresh runtime

In [ ]:
# Enqueue the stratified probe universe (40-60k domains balanced across classes)
universe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
n = ledger.enqueue(universe.to_dict('records'))
print('newly enqueued:', n)
print(ledger.summary())

In [ ]:
summary = tls_prober.run_collection(
    ledger,
    out_dir=f"{P['data']['collected']}/tls_probe",
    batch_size=2000, concurrency=100, logger=log)
print(summary)

In [ ]:
ledger.retire_exhausted()   # transient failures past the retry cap -> 'abandoned'
print(ledger.summary())
ledger.backup()

## Certificate Transparency (separate, slower, also resumable)

crt.sh is rate-sensitive. Run this in a second session rather than competing with the TLS probe for bandwidth.

In [ ]:
from src.collect import crtsh_client
from src.utils.io import ShardWriter
import time

todo = ledger.pending(limit=5000)
with ShardWriter(f"{P['data']['collected']}/crtsh", 'crtsh') as w:
    for d in todo:
        status, rows = crtsh_client.fetch(d)
        if status == 'success':
            w.add(crtsh_client.summarise(d, rows))
        time.sleep(0.5)   # be polite